# Chroma & Lanchain quickstart 

In this notebook we will try to see how to interact with chomadb and langchain in order to wrap all this into a script for the next activities 🤗

## Chroma docker container 

First you've must run chroma inside docker with the command below : 

```bash
docker run -d \
  --name chroma-db \
  -p 8001:8001 \
  -e ALLOW_RESET=true \
  -e ANONYMIZED_TELEMETRY=false \
  -e CHROMA_SERVER_AUTH_CREDENTIALS_ENABLE=false \
  -e CHROMA_SERVER_HTTP_PORT=8000 \
  -v "$(pwd)/data/chroma:/chroma/chroma" \
  --network=host \
  ghcr.io/chroma-core/chroma:latest
```

You should have a chroma container running, you can check it with de `docker ps` command. 


In [9]:
import chromadb
from chromadb.utils import embedding_functions
from chromadb.config import Settings

In [10]:
# Connect with no authentication on the port of your choice 
chroma_client = chromadb.HttpClient(host = 'localhost', 
                                    port = 8000)

# Note: Changed 'port = 8001' to 'port = 8000'

Follow the [official quickstart](https://docs.trychroma.com/docs/overview/getting-started) to create a collection named `document_store`, add document into it and try to query the collection like in the cells below 

> ⚠️ you will not have the same values as output since we use differents data 

In [11]:
# Creating a collection following the 'official quickstart':
collection1 = chroma_client.create_collection(name = "document_store")

In [12]:
# Adding document into the collection:
collection1.add(
    documents = [
        "This is a document about pineapple",
        "This is a document about oranges"],
    ids = ["id1", "id2"])

In [13]:
# Quering the collection:
results1 = collection1.query(
    query_texts=["What documents do I have?"],
    n_results = 2)

# Printing the results:
print(results1)

{'ids': [['id1', 'id2']], 'distances': [[1.5597879, 1.5680456]], 'embeddings': None, 'metadatas': [[None, None]], 'documents': [['This is a document about pineapple', 'This is a document about oranges']], 'uris': None, 'data': None, 'included': ['metadatas', 'documents', 'distances']}


After having created my first collection, I will be creating one more collection in order to determine if we really can create several different collections. You can find the other collection code below:

In [14]:
# Coding the chroma_client for collection2:
collection2 = chroma_client.create_collection(name = 'electronics_store')

collection2.add(
    documents = [
        "MacbookPro laptop",
        "Samsung Galaxy A4"],
    metadatas = [{"category": "computers"}, {"category": "phones"}], 
    ids = ["elec1", "elec2"])

# Note: I learned that metadatas is a dictionary for key-value pairs.

results2 = collection2.query(
    query_texts=["What electronics do I have?"],
    n_results = 2)

# Printing the results:
print(results2)

{'ids': [['elec1', 'elec2']], 'distances': [[1.2844863, 1.4292024]], 'embeddings': None, 'metadatas': [[{'category': 'computers'}, {'category': 'phones'}]], 'documents': [['MacbookPro laptop', 'Samsung Galaxy A4']], 'uris': None, 'data': None, 'included': ['metadatas', 'documents', 'distances']}


In [15]:
# Counting number of collections:
chroma_client.count_collections()

2

In [16]:
# Listing the existing collections:
chroma_client.list_collections()

[Collection(name=document_store), Collection(name=electronics_store)]

In [19]:
# Getting the 'ids':
collection1.get()['ids'][:5]

['id1', 'id2']

In [20]:
collection2.get()['ids'][:5]

['elec1', 'elec2']

## Langchain in a nutshell 

[LangChain](https://python.langchain.com/v0.1/docs/get_started/quickstart/) is a framework that simplifies building applications powered by language models, providing components for document handling, memory, agents, and chains to orchestrate complex workflows. 

Its primary purpose is to help developers create context-aware AI applications that can connect language models to external data sources and tools.RetryClaude can make mistakes. Please double-check responses. Let's take a look on how it's work. 

In [ ]:
# Importing the Ollama API:
from langchain_community.llms import Ollama

# Implement the correct model type:
llm = Ollama(model = "llama3.2")

In [ ]:
# Attempting a question:
response = llm.invoke("How can I get better at coding?")

print(response)

Improving your coding skills takes time, practice, and dedication. Here are some tips to help you get better at coding:

1. **Practice regularly**: The best way to learn coding is by writing code. Set aside time each week to work on projects or exercises.
2. **Start with the basics**: Make sure you have a solid understanding of the fundamentals, such as data types, variables, control structures, and functions.
3. **Choose a programming language**: Focus on one language at a time, rather than trying to learn multiple languages simultaneously.
4. **Join online communities**: Participate in online forums, social media groups, or Reddit communities related to coding. This will help you connect with other coders, get feedback on your code, and stay updated on industry trends.
5. **Find a mentor**: Having a experienced coder as a mentor can be incredibly helpful. They can provide guidance, support, and constructive feedback on your projects.
6. **Work on projects**: Apply your coding skills 

In [39]:
# Guiding Ollama's response with prompt template:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world class engineer with more than 20y of experience."),
    ("user", "{input}")])

In [40]:
chain = prompt | llm 

In [ ]:
# Printing to view how prompt template changed the original response:
response_w_guide = chain.invoke("How can I be better at coding?")

print(response_w_guide)

An excellent question, my friend! As an engineer with over 20 years of experience, I've seen many coders come and go, and I'm happy to share some advice on how to improve your coding skills.

First and foremost, it's essential to understand that coding is a skill that takes time and practice to develop. It's not something you can master overnight, but with dedication and persistence, you can become an exceptional coder.

Here are some tips that have worked for me throughout my career:

1. **Practice consistently**: Coding is like any other skill - the more you practice, the better you'll become. Set aside a specific time each day or week to work on coding projects, and make sure you stick to it.
2. **Learn from others**: Read code written by other developers, participate in online forums and communities, and learn from open-source projects. This will help you understand different coding styles, approaches, and best practices.
3. **Focus on fundamentals**: Don't skip the basics! Make su

### Parser 

LangChain's parsers are specialized tools that help extract structured data from unstructured text or LLM outputs. They convert raw text into usable formats like dictionaries, lists, or custom objects. Let's follow the doc and use `StrOutputParser` class 


In [46]:
# Implementing Parser by Following the instruction of the 'LangChain' website:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
chain = prompt | llm | output_parser

In [ ]:
# Printing response when Parser was implemented:
response_w_parser = chain.invoke("how can I be better at coding ?")

print(response_w_parser)

A question that gets to the heart of my passion - making technology work for people!

As an experienced engineer, I've seen many individuals struggle to improve their coding skills. But don't worry, it's a skill that can be developed with practice, patience, and persistence. Here are some tips to help you become better at coding:

1. **Practice consistently**: Coding is like any other skill - the more you use it, the more comfortable and proficient you'll become. Set aside dedicated time each day or week to work on small projects or exercises.
2. **Learn from others**: Study open-source code, read documentation, and observe how other developers approach problems. You can learn a lot by seeing how they think and solve issues.
3. **Focus on fundamentals**: Make sure you have a solid grasp of the basics: data structures, algorithms, computer science concepts, and programming paradigms (e.g., OOP, functional programming).
4. **Build projects**: Start with small, achievable projects that in

In [ ]:
# Checking if the answer is now str rather than a ChatMessage:
type(chain.invoke("how can I be better at coding ?"))

str

In this notebook we will not be interacting with the [Retrieval Chain](https://python.langchain.com/v0.1/docs/get_started/quickstart/) since we do not need web informtions, but you can check it from the doc 😎

In the next part let's see how to connect langchain and chroma [here](https://python.langchain.com/docs/integrations/vectorstores/chroma/#basic-initialization) 

In [65]:
!pip install -U "chromadb>=0.4.24" "langchain-chroma>=0.1.1" "langchain>=0.1.14" "langchain-ollama"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached chromadb-1.0.4-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.9 kB)
  Using cached langchain_chroma-0.2.2-py3-none-any.whl.metadata (1.3 kB)
  Using cached langchain-0.3.23-py3-none-any.whl.metadata (7.8 kB)
  Using cached chromadb-0.6.3-py3-none-any.whl.metadata (6.8 kB)
Using cached langchain_chroma-0.2.2-py3-none-any.whl (11 kB)
Using cached chromadb-0.6.3-py3-none-any.whl (611 kB)
Using cached langchain-0.3.23-py3-none-any.whl (1.0 MB)


In [2]:
# Changing code from 'OpenAi' to 'Ollama' model:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="llama3.2")

vector_store = Chroma(
    collection_name = "example_collection",
    embedding_function = embeddings,
    persist_directory = "./chroma")

### Your mini mission 

Now that you have the basics your mission is : code a function named batch_upload_document like below and upload this [PDF](https://arxiv.org/abs/1706.03762) inside your collection and do a simple query 

```python
def batch_upload_documents(
    files: List[UploadFile] = File(...),
    title_prefix: str = Form("Document"),
    source: str = Form(""),
    author: str = Form(""),
    tags: str = Form(""),
    chunk_size: int = Form(1000, description="Size of text chunks for PDF documents"),
    chunk_overlap: int = Form(100, description="Overlap between text chunks for PDF documents"),
    index_id: Optional[str] = Form(None, description="Optional index ID to add documents to")):
    """
    Upload multiple documents to the knowledge base and optionally add to an index.
    Handles both text files and PDFs.
    
    For PDF files, the document will be split into chunks using LangChain's text splitter.
    """
    #TODO: code here 
    pass 
```

You should see this kind of results below 🥸

Note to Professor: This section is not done as I have run out of time. It is not completed NOT because of issues, but a lack of time and how slow I am at coding in general.

In [ ]:
results = collection.query(
    query_texts=["attention"],
    n_results=5
)
results

{'ids': [['a39e9252-9f9d-4787-98af-e428a1a8088f',
   'dcff76f7-450b-4315-a348-20368b50afcc',
   'f9b79128-b5e9-4094-afc9-649096630d00',
   'db5de2e2-4281-48d8-bc51-5d928e7c3720',
   '70c60f56-8215-4e21-86be-e6e6ee061ea9']],
 'distances': [[0.9198479056358337,
   0.9198479056358337,
   0.9216511249542236,
   0.9216515948930766,
   0.9234551787376404]],
 'embeddings': None,
 'metadatas': [[{'author': '',
    'created_at': '2025-03-11T00:08:04.573912',
    'description': 'PDF page 9 uploaded as part of batch on 2025-03-11',
    'filename': 'attention.pdf_page9_chunk37',
    'id': 'a39e9252-9f9d-4787-98af-e428a1a8088f',
    'source': 'attention.pdf',
    'tags': 'pdf',
    'title': 'Document 1: attention.pdf (Page 9, Chunk 37)',
    'updated_at': '2025-03-11T00:08:04.573923'},
   {'author': '',
    'created_at': '2025-03-11T00:07:29.513343',
    'description': 'PDF page 9 uploaded as part of batch on 2025-03-11',
    'filename': 'attention.pdf_page9_chunk37',
    'id': 'dcff76f7-450b-4315-